# Notebook 06: Cross-Domain Experiments
## TSFM Industrial PdM Benchmark - ICATH Conference

**Author:** Yassire Ammouri  
**Purpose:** Evaluate cross-domain generalization (train on A, test on B)  
**Key Question:** How well do TSFMs transfer across industrial domains?  
**Expected Runtime:** 2-4 hours on T4 GPU

### Step 1: Setup and Imports

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import json
import time
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Add project source to path
PROJECT_DIR = Path("/content/tsfm-pdm-benchmark")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from models import get_model, list_models
from evaluation.metrics import compute_all_metrics

# Paths
PROCESSED_DIR = PROJECT_DIR / "data/processed"
RESULTS_DIR = PROJECT_DIR / "results/cross_domain"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Configuration

In [ ]:
# Load configs
with open(PROJECT_DIR / "config/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Models to evaluate
MODELS = ["moment", "chronos", "patchtst"]

# Datasets for cross-domain (need at least 2 different domains)
DATASETS = ["cmapss/FD001", "cmapss/FD002", "cmapss/FD003", "cmapss/FD004"]

TASK = "forecasting"
HORIZON = config['preprocessing']['forecast_horizon']

print("Cross-Domain Configuration:")
print(f"  Models: {MODELS}")
print(f"  Datasets: {DATASETS}")
print(f"  Transfer pairs: {len(DATASETS) * (len(DATASETS) - 1)}")
print(f"  Total experiments: {len(MODELS) * len(DATASETS) * (len(DATASETS) - 1)}")

### Step 3: Cross-Domain Experiment Function

In [ ]:
def run_cross_domain(model_name, train_dataset, test_dataset, 
                     train_data_path, test_data_path,
                     task="forecasting", horizon=96, device="cuda"):
    """Run cross-domain evaluation: train on A, test on B"""
    print(f"\n{'='*60}")
    print(f"Cross-Domain: {model_name} | Train: {train_dataset} -> Test: {test_dataset}")
    print(f"{'='*60}")
    
    # Load data
    train_data = torch.load(train_data_path / "processed_data.pt")
    test_data = torch.load(test_data_path / "processed_data.pt")
    
    X_train = train_data['train_X']
    y_train = train_data['train_y']
    X_test = test_data['test_X']
    y_test = test_data['test_y']
    
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")
    
    # Load model
    model = get_model(model_name, device=device)
    model.load_model()
    
    # Run zero-shot inference (no domain adaptation)
    print("Running zero-shot inference (no domain adaptation)...")
    start_time = time.time()
    
    results = model.zero_shot(X_test, horizon=horizon)
    predictions = results['predictions']
    
    inference_time = time.time() - start_time
    
    # Handle shape mismatches
    if predictions.shape != y_test.shape:
        min_len = min(predictions.shape[1], y_test.shape[1])
        predictions = predictions[:, :min_len]
        y_test = y_test[:, :min_len]
        print(f"Shape adjusted to: {predictions.shape}")
    
    # Compute metrics
    metrics = compute_all_metrics(
        y_test.numpy() if isinstance(y_test, torch.Tensor) else y_test,
        predictions,
        task=task
    )
    
    print(f"Cross-Domain Results: {metrics}")
    
    return {
        'model': model_name,
        'train_dataset': train_dataset,
        'test_dataset': test_dataset,
        'task': task,
        'scenario': 'cross_domain',
        'metrics': metrics,
        'inference_time': inference_time,
        'timestamp': datetime.now().isoformat()
    }

### Step 4: Run Cross-Domain Transfer Matrix

In [ ]:
all_results = []
transfer_matrix = {}
errors = []

print(f"\nStarting cross-domain experiments...")
print(f"Models: {MODELS}")
print(f"Datasets: {DATASETS}")
print(f"Total: {len(MODELS) * len(DATASETS) * (len(DATASETS) - 1)} experiments\n")

for model_name in tqdm(MODELS, desc="Models"):
    model_matrix = {}
    
    for train_ds in DATASETS:
        train_path = PROCESSED_DIR / train_ds
        if not train_path.exists():
            continue
        
        for test_ds in DATASETS:
            if train_ds == test_ds:
                continue  # Skip same-domain (already in zero-shot)
            
            test_path = PROCESSED_DIR / test_ds
            if not test_path.exists():
                continue
            
            try:
                result = run_cross_domain(
                    model_name=model_name,
                    train_dataset=train_ds,
                    test_dataset=test_ds,
                    train_data_path=train_path,
                    test_data_path=test_path,
                    task=TASK,
                    horizon=HORIZON,
                    device=device
                )
                all_results.append(result)
                model_matrix[f"{train_ds}->{test_ds}"] = result['metrics'].get('mae', float('nan'))
                
            except Exception as e:
                print(f"Error: {model_name} {train_ds}->{test_ds}: {e}")
                errors.append({
                    'model': model_name,
                    'train_dataset': train_ds,
                    'test_dataset': test_ds,
                    'error': str(e)
                })
    
    transfer_matrix[model_name] = model_matrix
    
    # Clear GPU memory
    torch.cuda.empty_cache()

print(f"\nExperiments complete!")
print(f"Successful: {len(all_results)}")
print(f"Errors: {len(errors)}")

### Step 5: Save Results

In [ ]:
# Create results DataFrame
df_results = pd.DataFrame(all_results)

# Save to CSV
df_results.to_csv(RESULTS_DIR / "cross_domain_results.csv", index=False)
print(f"Results saved to: {RESULTS_DIR / 'cross_domain_results.csv'}")

# Save transfer matrix
with open(RESULTS_DIR / "transfer_matrix.json", 'w') as f:
    json.dump(transfer_matrix, f, indent=2)
print(f"Transfer matrix saved to: {RESULTS_DIR / 'transfer_matrix.json'}")

# Save errors
if errors:
    df_errors = pd.DataFrame(errors)
    df_errors.to_csv(RESULTS_DIR / "cross_domain_errors.csv", index=False)
    print(f"Errors saved to: {RESULTS_DIR / 'cross_domain_errors.csv'}")

### Step 6: Display Transfer Matrix

In [ ]:
print("=" * 80)
print("CROSS-DOMAIN TRANSFER MATRIX (MAE)")
print("=" * 80)

for model_name, matrix in transfer_matrix.items():
    print(f"\n{model_name}:")
    for pair, mae in matrix.items():
        print(f"  {pair}: {mae:.4f}")

### Step 7: Visualize Transfer Matrix

In [ ]:
if len(transfer_matrix) > 0:
    # Create heatmap for first model
    first_model = list(transfer_matrix.keys())[0]
    matrix = transfer_matrix[first_model]
    
    # Extract unique datasets
    datasets = sorted(list(set(
        [pair.split('->')[0] for pair in matrix.keys()] + 
        [pair.split('->')[1] for pair in matrix.keys()]
    )))
    
    # Create matrix
    n = len(datasets)
    heatmap_data = np.full((n, n), np.nan)
    
    for i, train_ds in enumerate(datasets):
        for j, test_ds in enumerate(datasets):
            if train_ds == test_ds:
                continue
            key = f"{train_ds}->{test_ds}"
            if key in matrix:
                heatmap_data[i, j] = matrix[key]
    
    # Plot
    fig, axes = plt.subplots(1, len(transfer_matrix), figsize=(8*len(transfer_matrix), 7))
    if len(transfer_matrix) == 1:
        axes = [axes]
    
    for idx, (model_name, matrix) in enumerate(transfer_matrix.items()):
        # Recreate matrix for this model
        heatmap_data = np.full((n, n), np.nan)
        for i, train_ds in enumerate(datasets):
            for j, test_ds in enumerate(datasets):
                if train_ds == test_ds:
                    continue
                key = f"{train_ds}->{test_ds}"
                if key in matrix:
                    heatmap_data[i, j] = matrix[key]
        
        # Short names for display
        short_names = [ds.split('/')[-1] if '/' in ds else ds for ds in datasets]
        
        mask = np.eye(n, dtype=bool)
        
        sns.heatmap(
            heatmap_data,
            annot=True,
            fmt=".3f",
            cmap="RdYlGn_r",
            ax=axes[idx],
            xticklabels=short_names,
            yticklabels=short_names,
            mask=mask,
            linewidths=0.5,
            cbar_kws={'label': 'MAE'}
        )
        
        axes[idx].set_title(f'{model_name}', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Test Dataset', fontsize=12)
        axes[idx].set_ylabel('Train Dataset', fontsize=12)
    
    plt.suptitle('Cross-Domain Transfer Performance', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cross_domain_heatmap.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Figure saved to: {RESULTS_DIR / 'cross_domain_heatmap.png'}")

### Step 8: Analyze Domain Gap

In [ ]:
if len(df_results) > 0:
    print("=" * 80)
    print("DOMAIN GAP ANALYSIS")
    print("=" * 80)
    
    # Compare cross-domain vs same-domain (zero-shot)
    zero_shot_file = PROJECT_DIR / "results/zero_shot/zero_shot_results.csv"
    
    if zero_shot_file.exists():
        df_zero = pd.read_csv(zero_shot_file)
        
        # Extract metrics
        cross_data = []
        for _, row in df_results.iterrows():
            cross_data.append({
                'model': row['model'],
                'test_dataset': row['test_dataset'],
                'mae_cross': row['metrics'].get('mae', float('nan'))
            })
        
        df_cross_extract = pd.DataFrame(cross_data)
        
        print("\nCross-domain MAE by test dataset:")
        print(df_cross_extract.groupby(['model', 'test_dataset'])['mae_cross'].mean().to_string())
        
    print("\n" + "=" * 80)
    print("Key Finding: Cross-domain performance degradation indicates")
    print("that TSFMs struggle with distribution shift between industrial domains.")
    print("This supports the need for federated learning approaches.")
    print("=" * 80)

### Step 9: Save to Google Drive

In [ ]:
import shutil

DRIVE_RESULTS = Path("/content/drive/MyDrive/ICATH_Results/cross_domain")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# Copy results
for f in RESULTS_DIR.glob("*"):
    if f.is_file():
        shutil.copy(f, DRIVE_RESULTS / f.name)

print(f"Results backed up to: {DRIVE_RESULTS}")
print("\nNext: Run Notebook 07 (Analysis & Visualization)")